# HAM10000 — Evaluación de calidad de imágenes sintéticas

Evalúa las fuentes de imágenes sintéticas de melanoma respecto a la distribución real.

| Fuente | N | Resolución | Método |
|---|---|---|---|
| `real_mel` | 801 | variable | Referencia HAM10000 |
| `textual_inversion` | 4500 | 512×512 | SD TI token `<mel-skin>` |
| `img2img` | 2403 | 512×512 | SD img2img variaciones de reales |
| `gan_final` | 5000 | 64×64 | WGAN-GP — guardado con PIL |
| `lora` | 4500 | 512×512 | DreamBooth-LoRA rank=32, 6000 steps |
| `derm_s040` | ~2400 | 512×512 | Derm-T2IM img2img strength=0.40 |
| `derm_s005` | ~2400 | 512×512 | Derm-T2IM img2img strength=0.05 |

**Métricas:**
- **FID** (Fréchet Inception Distance): distancia entre distribuciones real y sintética — menor es mejor
- **IS** (Inception Score): calidad y diversidad de las sintéticas — mayor es mejor
- **Filtro calidad**: % de imágenes que pasan un filtro de calidad visual mínima
- **Grillas visuales**: inspección manual de muestras aleatorias

⚠️ FID usa features de InceptionV3 (ImageNet). Para imágenes dermoscópicas los valores absolutos
deben interpretarse comparativamente entre fuentes, no como valores absolutos.

In [1]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_CUDA = DEVICE.type == 'cuda'
print(f'IN_COLAB={IN_COLAB}  device={DEVICE}')

IN_COLAB=True  device=cpu


In [ ]:
from pathlib import Path

if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE_ROOT   = Path('/content/drive/MyDrive/ham10000-augmentation')
    PROJECT_ROOT = DRIVE_ROOT
    SYNTH_ROOT   = DRIVE_ROOT / 'synthetic'
    GAN_ZIP      = next(DRIVE_ROOT.glob('gan_generated*.zip'), None)
    GAN_FINAL_ZIP = next(DRIVE_ROOT.glob('gan_final*.zip'), None)
    LORA_ZIP      = next(DRIVE_ROOT.glob('lora*.zip'), None)
else:
    PROJECT_ROOT  = Path.cwd()
    SYNTH_ROOT    = PROJECT_ROOT / 'data/synthetic'
    GAN_ZIP       = next(PROJECT_ROOT.glob('gan_generated*.zip'), None)
    GAN_FINAL_ZIP = next(PROJECT_ROOT.glob('gan_final*.zip'), None)
    LORA_ZIP      = next(PROJECT_ROOT.glob('lora*.zip'), None)

SPLITS_DIR    = PROJECT_ROOT / 'data/processed/splits'
IMAGES_DIR    = PROJECT_ROOT / 'data/processed/images'
GAN_FINAL_DIR = SYNTH_ROOT / 'gan_final'
LORA_DIR      = SYNTH_ROOT / 'lora'
DERM_040_DIR  = SYNTH_ROOT / 'derm_s040'
DERM_005_DIR  = SYNTH_ROOT / 'derm_s005'
EVAL_OUT      = PROJECT_ROOT / 'reports/quality_evaluation'
EVAL_OUT.mkdir(parents=True, exist_ok=True)

print(f'Sintéticas:   {SYNTH_ROOT}')
print(f'GAN final:    {GAN_FINAL_DIR}  (existe: {GAN_FINAL_DIR.exists()})')
print(f'LoRA:         {LORA_DIR}  (existe: {LORA_DIR.exists()})')
print(f'Derm s=0.40:  {DERM_040_DIR}  (existe: {DERM_040_DIR.exists()})')
print(f'Derm s=0.05:  {DERM_005_DIR}  (existe: {DERM_005_DIR.exists()})')
print(f'Salida:       {EVAL_OUT}')

In [3]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

try:
    import torch_fidelity; print(f'torch_fidelity ok')
except ImportError:
    pip_install('torch-fidelity'); import torch_fidelity

try:
    import cv2; print(f'cv2 ok')
except ImportError:
    pip_install('opencv-python-headless'); import cv2

try:
    import pandas as pd; print(f'pandas ok')
except ImportError:
    pip_install('pandas'); import pandas as pd

import numpy as np
import json, random, shutil, zipfile
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
print('Dependencias listas')

cv2 ok
pandas ok
Dependencias listas


## Estado de la evaluación
Ejecuta esta celda para ver qué bloques ya están completos.

In [4]:
MARKERS = {
    'GAN extraído':      EVAL_OUT / 'gan_extracted.json',
    'GAN final extraído': EVAL_OUT / 'gan_final_extracted.json',
    'LoRA extraído':     EVAL_OUT / 'lora_extracted.json',
    'Ref real copiada':  EVAL_OUT / 'real_ref_ready.json',
    'FID/IS completo':   EVAL_OUT / 'fid_results.json',
    'Filtro calidad':    EVAL_OUT / 'quality_filter.json',
    'Grillas visuales':  EVAL_OUT / 'grids_done.txt',
}

for name, path in MARKERS.items():
    if path.exists() and path.suffix == '.json':
        try:
            info = json.loads(path.read_text())
            status = '✅'
            extra  = str(info) if len(str(info)) < 80 else ''
        except Exception:
            status, extra = '✅', ''
    else:
        status, extra = ('✅', '') if path.exists() else ('⬜', '')
    print(f'  {status} {name}  {extra}')


  ✅ GAN extraído  {'gan_64': 1000, 'gan_640': 4000}
  ✅ GAN final extraído  {'n': 5000}
  ✅ LoRA extraído  {'n': 4500}
  ✅ Ref real copiada  {'count': 801}
  ✅ FID/IS completo  
  ⬜ Filtro calidad  
  ⬜ Grillas visuales  


## Paso 1 — Preparar directorios de imágenes

In [5]:
# --- Extraer GAN ZIP original (gan_64 / gan_640) ---
gan_marker = EVAL_OUT / 'gan_extracted.json'
GAN_64_DIR  = SYNTH_ROOT / 'gan_64'
GAN_640_DIR = SYNTH_ROOT / 'gan_640'

if gan_marker.exists():
    info = json.loads(gan_marker.read_text())
    print(f'GAN original ya extraído — gan_64: {info["gan_64"]}  gan_640: {info["gan_640"]}')
else:
    if GAN_ZIP and GAN_ZIP.exists():
        GAN_64_DIR.mkdir(parents=True, exist_ok=True)
        GAN_640_DIR.mkdir(parents=True, exist_ok=True)
        print('Extrayendo GAN ZIP original...')
        import struct
        with zipfile.ZipFile(GAN_ZIP) as zf:
            for member in zf.infolist():
                if not member.filename.endswith('.png'):
                    continue
                data = zf.read(member.filename)
                w = struct.unpack('>I', data[16:20])[0]
                h = struct.unpack('>I', data[20:24])[0]
                fname = Path(member.filename).name
                (GAN_64_DIR if (w == 64 and h == 64) else GAN_640_DIR).joinpath(fname).write_bytes(data)
        n64  = len(list(GAN_64_DIR.glob('*.png')))
        n640 = len(list(GAN_640_DIR.glob('*.png')))
        gan_marker.write_text(json.dumps({'gan_64': n64, 'gan_640': n640}, indent=2))
        print(f'Listo — gan_64: {n64}  gan_640: {n640}')
    else:
        print('GAN ZIP original no encontrado — omitido')

# --- Extraer gan_final.zip ---
gfinal_marker = EVAL_OUT / 'gan_final_extracted.json'
if gfinal_marker.exists():
    info = json.loads(gfinal_marker.read_text())
    print(f'GAN final ya extraído — {info["n"]} imágenes en {GAN_FINAL_DIR}')
else:
    GAN_FINAL_DIR.mkdir(parents=True, exist_ok=True)
    if len(list(GAN_FINAL_DIR.glob('*.png'))) >= 10:
        n = len(list(GAN_FINAL_DIR.glob('*.png')))
        gfinal_marker.write_text(json.dumps({'n': n}, indent=2))
        print(f'GAN final ya en disco — {n} imágenes')
    elif GAN_FINAL_ZIP and GAN_FINAL_ZIP.exists():
        print('Extrayendo gan_final.zip...')
        with zipfile.ZipFile(GAN_FINAL_ZIP) as zf:
            for m in zf.infolist():
                if m.filename.endswith('.png'):
                    (GAN_FINAL_DIR / Path(m.filename).name).write_bytes(zf.read(m.filename))
        n = len(list(GAN_FINAL_DIR.glob('*.png')))
        gfinal_marker.write_text(json.dumps({'n': n}, indent=2))
        print(f'Listo — {n} imágenes GAN final')
    else:
        print('gan_final.zip no encontrado — omitido')

# --- Extraer lora.zip ---
lora_marker = EVAL_OUT / 'lora_extracted.json'
if lora_marker.exists():
    info = json.loads(lora_marker.read_text())
    print(f'LoRA ya extraído — {info["n"]} imágenes en {LORA_DIR}')
else:
    LORA_DIR.mkdir(parents=True, exist_ok=True)
    if len(list(LORA_DIR.glob('*.jpg'))) >= 10:
        n = len(list(LORA_DIR.glob('*.jpg')))
        lora_marker.write_text(json.dumps({'n': n}, indent=2))
        print(f'LoRA ya en disco — {n} imágenes')
    elif LORA_ZIP and LORA_ZIP.exists():
        print('Extrayendo lora.zip...')
        with zipfile.ZipFile(LORA_ZIP) as zf:
            for m in zf.infolist():
                if m.filename.lower().endswith('.jpg'):
                    (LORA_DIR / Path(m.filename).name).write_bytes(zf.read(m.filename))
        n = len(list(LORA_DIR.glob('*.jpg')))
        lora_marker.write_text(json.dumps({'n': n}, indent=2))
        print(f'Listo — {n} imágenes LoRA')
    else:
        print('lora.zip no encontrado — omitido')


GAN original ya extraído — gan_64: 1000  gan_640: 4000
GAN final ya extraído — 5000 imágenes en /content/drive/MyDrive/ham10000-augmentation/synthetic/gan_final
LoRA ya extraído — 4500 imágenes en /content/drive/MyDrive/ham10000-augmentation/synthetic/lora


In [6]:
# --- Preparar directorio de referencia real (solo mel de train) ---
ref_marker = EVAL_OUT / 'real_ref_ready.json'
REAL_MEL_DIR = Path('/content/real_mel_ref') if IN_COLAB \
               else PROJECT_ROOT / 'data/synthetic/real_mel_ref'

# En Colab /content/ se borra al reiniciar — verificar que las imágenes existen, no solo el marker
n_existing = len(list(REAL_MEL_DIR.glob('*.jpg'))) if REAL_MEL_DIR.exists() else 0
already_done = ref_marker.exists() and n_existing > 0

if already_done:
    print(f'Referencia real lista — {n_existing} imágenes en {REAL_MEL_DIR}')
else:
    REAL_MEL_DIR.mkdir(parents=True, exist_ok=True)

    if IN_COLAB:
        zip_path  = PROJECT_ROOT / 'classification_data.zip'
        extracted = Path('/content/classification_data')
        if not extracted.exists():
            print('Extrayendo classification_data.zip...')
            import subprocess
            subprocess.check_call(['unzip', '-q', str(zip_path), '-d', str(extracted)])

        hits = list(extracted.rglob('train.csv'))
        if not hits:
            raise FileNotFoundError(f'train.csv no encontrado en {extracted}')
        split_base = hits[0].parent

        img_candidates = [extracted / 'images', split_base.parent / 'images']
        img_base = next((p for p in img_candidates if p.exists()), None)
        if img_base is None:
            raise FileNotFoundError('Carpeta images/ no encontrada en el zip extraído')
    else:
        split_base = SPLITS_DIR
        img_base   = PROJECT_ROOT / 'data/HAM10000_images'
        if not img_base.exists():
            img_base = PROJECT_ROOT / 'data/processed/images'

    train_df = pd.read_csv(split_base / 'train.csv')
    name_col = 'image_id' if 'image_id' in train_df.columns else 'image_path'
    mel_val  = 1 if train_df['label'].dtype != object else 'mel'
    mel_rows = train_df[train_df['label'] == mel_val]

    copied = 0
    for _, row in mel_rows.iterrows():
        fname = Path(str(row[name_col])).name
        if not fname.endswith('.jpg'):
            fname += '.jpg'
        src = img_base / fname
        if src.exists():
            shutil.copy2(src, REAL_MEL_DIR / fname)
            copied += 1

    ref_marker.write_text(json.dumps({'count': copied}, indent=2))
    print(f'✅ {copied} imágenes de melanoma real → {REAL_MEL_DIR}')


Extrayendo classification_data.zip...
✅ 801 imágenes de melanoma real → /content/real_mel_ref


## Paso 2 — FID e IS por fuente

FID mide la distancia entre la distribución de features Inception de las imágenes reales
y las sintéticas. IS mide calidad y diversidad de las sintéticas.

Este bloque tarda ~5–10 min por fuente en CPU.

In [ ]:
from torch_fidelity import calculate_metrics

fid_marker = EVAL_OUT / 'fid_results.json'

SOURCES = {
    'textual_inversion': SYNTH_ROOT / 'textual_inversion',
    'img2img':           SYNTH_ROOT / 'img2img',
    'gan_final':         GAN_FINAL_DIR,
    'lora':              LORA_DIR,
    'derm_s040':         DERM_040_DIR,
    'derm_s005':         DERM_005_DIR,
}

# En Colab: copiar fuentes de Drive FUSE a /content/ — torch_fidelity no lee bien rutas FUSE
if IN_COLAB:
    LOCAL_FID  = Path('/content/fid_sources')
    LOCAL_REAL = Path('/content/fid_real')
    local_sources = {}
    for name, src_dir in SOURCES.items():
        local = LOCAL_FID / name
        if src_dir.exists():
            exts  = ['.jpg', '.jpeg', '.png']
            n_src = len([p for p in src_dir.glob('*.*') if p.suffix.lower() in exts])
            n_loc = len([p for p in local.glob('*.*') if p.suffix.lower() in exts]) if local.exists() else 0
            if n_loc < n_src:
                print(f'Copiando {name} a /content/ ({n_src} imgs)...')
                if local.exists():
                    shutil.rmtree(local)
                shutil.copytree(str(src_dir), str(local))
            local_sources[name] = local
        else:
            local_sources[name] = src_dir

    n_real_loc = len(list(LOCAL_REAL.glob('*.jpg'))) if LOCAL_REAL.exists() else 0
    n_real_src = len(list(REAL_MEL_DIR.glob('*.jpg')))
    if n_real_loc < n_real_src:
        print(f'Copiando real_mel_ref a /content/ ({n_real_src} imgs)...')
        if LOCAL_REAL.exists():
            shutil.rmtree(LOCAL_REAL)
        shutil.copytree(str(REAL_MEL_DIR), str(LOCAL_REAL))
    fid_real_dir = LOCAL_REAL
    SOURCES = local_sources
    print('Fuentes locales listas\n')
else:
    fid_real_dir = REAL_MEL_DIR

# ResizedDataset — evita errores por imágenes de resolución variable
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

class ResizedDataset(Dataset):
    def __init__(self, folder, size=(299, 299)):
        exts = {'.jpg', '.jpeg', '.png'}
        self.paths = sorted(p for p in Path(folder).glob('*.*') if p.suffix.lower() in exts)
        self.size  = size
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB').resize(self.size, Image.BILINEAR)
            return TF.pil_to_tensor(img)
        except Exception:
            return torch.zeros(3, *self.size, dtype=torch.uint8)

# Cargar resultados existentes (incremental — solo calcula fuentes nuevas)
fid_results = json.loads(fid_marker.read_text()) if fid_marker.exists() else {}
pending = {k: v for k, v in SOURCES.items() if k not in fid_results or 'error' in fid_results.get(k, {})}

if not pending:
    print('FID/IS ya calculados para todas las fuentes:')
    for src, m in fid_results.items():
        if 'fid' in m:
            print(f'  {src:25s}  FID={m["fid"]:.1f}  IS={m["is_mean"]:.3f}±{m["is_std"]:.3f}  N={m["n_synth"]}')
else:
    real_ds = ResizedDataset(fid_real_dir)
    print(f'Calculando {len(pending)} fuentes (cuda={USE_CUDA})...\n')
    for name, src_dir in pending.items():
        if not src_dir.exists():
            print(f'  {name}: directorio no encontrado — omitido')
            continue
        synth_ds = ResizedDataset(src_dir)
        if len(synth_ds) < 10:
            print(f'  {name}: muy pocas imágenes ({len(synth_ds)}) — omitido')
            continue
        print(f'Calculando FID/IS para {name} ({len(synth_ds)} imágenes)...')
        try:
            m = calculate_metrics(
                input1=real_ds, input2=synth_ds,
                cuda=USE_CUDA, fid=True, isc=True, verbose=False,
            )
            fid_results[name] = {
                'fid':     round(m['frechet_inception_distance'], 2),
                'is_mean': round(m['inception_score_mean'], 4),
                'is_std':  round(m['inception_score_std'], 4),
                'n_synth': len(synth_ds),
            }
            print(f'  → FID={fid_results[name]["fid"]:.1f}  IS={fid_results[name]["is_mean"]:.3f}±{fid_results[name]["is_std"]:.3f}')
        except Exception as e:
            print(f'  ERROR en {name}: {e}')
            fid_results[name] = {'error': str(e)}
        fid_marker.write_text(json.dumps(fid_results, indent=2))

    print('\nGuardado en', fid_marker)

print('\nResumen:')
for src, m in fid_results.items():
    if 'fid' in m:
        print(f'  {src:25s}  FID={m["fid"]:.1f}  IS={m["is_mean"]:.3f}  N={m["n_synth"]}')
    elif 'error' in m:
        print(f'  {src:25s}  ❌ error')

In [11]:
from torch_fidelity import calculate_metrics
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

class ResizedDataset(Dataset):
    def __init__(self, folder, size=(299, 299)):
        exts = {'.jpg', '.jpeg', '.png'}
        self.paths = sorted(p for p in Path(folder).glob('*.*') if p.suffix.lower() in exts)
        self.size  = size

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB').resize(self.size, Image.BILINEAR)
            return TF.pil_to_tensor(img)
        except Exception:
            return torch.zeros(3, *self.size, dtype=torch.uint8)

fid_marker  = EVAL_OUT / 'fid_results.json'
fid_results = json.loads(fid_marker.read_text()) if fid_marker.exists() else {}

pending = {k: v for k, v in {
    'lora': Path('/content/fid_sources/lora'),
}.items() if k not in fid_results or 'error' in fid_results.get(k, {})}

if not pending:
    print('lora ya calculado')
else:
    real_ds = ResizedDataset(Path('/content/fid_real'))
    for name, src_dir in pending.items():
        synth_ds = ResizedDataset(src_dir)
        print(f'Calculando FID/IS para {name} ({len(synth_ds)} imgs)...')
        m = calculate_metrics(input1=real_ds, input2=synth_ds, cuda=USE_CUDA, fid=True, isc=True, verbose=False)
        fid_results[name] = {
            'fid':     round(m['frechet_inception_distance'], 2),
            'is_mean': round(m['inception_score_mean'], 4),
            'is_std':  round(m['inception_score_std'], 4),
            'n_synth': len(synth_ds),
        }
        fid_marker.write_text(json.dumps(fid_results, indent=2))
        print(f'  → FID={fid_results[name]["fid"]:.1f}  IS={fid_results[name]["is_mean"]:.3f}')

print('\nResumen completo:')
for src, m in fid_results.items():
    if 'fid' in m:
        print(f'  {src:25s}  FID={m["fid"]:.1f}  IS={m["is_mean"]:.3f}  N={m["n_synth"]}')

Calculando FID/IS para lora (4501 imgs)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


  → FID=121.7  IS=3.430

Resumen completo:
  textual_inversion          FID=272.9  IS=3.374  N=4500
  img2img                    FID=170.6  IS=3.374  N=2403
  gan_64                     FID=223.2  IS=3.374  N=1000
  gan_final                  FID=220.8  IS=3.374  N=5000
  lora                       FID=121.7  IS=3.430  N=4501
  gan_640                    FID=311.6  IS=3.374  N=4000


## Paso 3 — Filtro de calidad visual

Detecta imágenes que probablemente son artefactos o generaciones fallidas usando:
- **Brillo**: descarta imágenes demasiado oscuras o quemadas
- **Contraste**: descarta imágenes sin variación (color sólido, ruido uniforme)
- **Tono piel**: verifica presencia mínima de tonos piel/lesión en HSV

In [ ]:
filter_marker = EVAL_OUT / 'quality_filter.json'

def quality_score(img_path: Path, size: int = 224):
    try:
        img = np.array(Image.open(img_path).convert('RGB').resize((size, size)))
        hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
        mean_v  = float(hsv[:, :, 2].mean())
        std_rgb = float(img.std(axis=(0, 1)).mean())
        h = hsv[:, :, 0]; s = hsv[:, :, 1]; v = hsv[:, :, 2]
        skin_mask  = ((h <= 25) | (h >= 150)) & (s > 20) & (v > 20)
        skin_ratio = float(skin_mask.mean())
        passes = (20 < mean_v < 235) and (std_rgb >= 8) and (skin_ratio >= 0.02)
        return passes, {'mean_v': round(mean_v, 1), 'std_rgb': round(std_rgb, 2),
                        'skin_ratio': round(skin_ratio, 4)}
    except Exception as e:
        return False, {'error': str(e)}


ALL_SOURCES = {
    'real_mel':           REAL_MEL_DIR,
    'textual_inversion':  SYNTH_ROOT / 'textual_inversion',
    'img2img':            SYNTH_ROOT / 'img2img',
    'gan_final':          GAN_FINAL_DIR,
    'lora':               LORA_DIR,
    'derm_s040':          DERM_040_DIR,
    'derm_s005':          DERM_005_DIR,
}

# Cargar resultados existentes (incremental)
filter_results = json.loads(filter_marker.read_text()) if filter_marker.exists() else {}
pending_filter = {k: v for k, v in ALL_SOURCES.items() if k not in filter_results}

if not pending_filter:
    print('Filtro ya calculado para todas las fuentes:')
    for src, stats in filter_results.items():
        print(f'  {src:25s}  {stats["passed"]}/{stats["total"]} pasan ({stats["pass_pct"]:.1f}%)')
else:
    print(f'Calculando filtro para {len(pending_filter)} fuentes nuevas...\n')
    for name, src_dir in pending_filter.items():
        if not src_dir.exists():
            print(f'  {name}: directorio no encontrado — omitido')
            continue
        exts  = ['*.jpg', '*.jpeg', '*.png']
        paths = [p for ext in exts for p in src_dir.glob(ext)]
        if not paths:
            continue
        sample = random.Random(42).sample(paths, min(500, len(paths)))
        passed = 0
        per_image_stats = []
        print(f'Filtrando {name} ({len(sample)} muestras)...')
        for p in sample:
            ok, stats = quality_score(p)
            if ok: passed += 1
            per_image_stats.append(stats)
        valid = [s for s in per_image_stats if 'error' not in s]
        agg   = {}
        if valid:
            agg['mean_v_avg']     = round(float(np.mean([s['mean_v'] for s in valid])), 1)
            agg['std_rgb_avg']    = round(float(np.mean([s['std_rgb'] for s in valid])), 2)
            agg['skin_ratio_avg'] = round(float(np.mean([s['skin_ratio'] for s in valid])), 4)
        filter_results[name] = {
            'total': len(sample), 'passed': passed,
            'pass_pct': round(100 * passed / len(sample), 1), **agg,
        }
        print(f'  → {passed}/{len(sample)} pasan ({filter_results[name]["pass_pct"]:.1f}%)')
    filter_marker.write_text(json.dumps(filter_results, indent=2))
    print('\nGuardado en', filter_marker)

print('\nResumen filtro:')
for src, stats in filter_results.items():
    print(f'  {src:25s}  {stats["passed"]}/{stats["total"]} ({stats["pass_pct"]:.1f}%)  skin_ratio={stats.get("skin_ratio_avg","?")}')

## Paso 4 — Grillas visuales de muestras aleatorias

In [ ]:
grids_marker = EVAL_OUT / 'grids_done.txt'

GRID_SOURCES = {
    'real_mel':          (REAL_MEL_DIR,                  ['*.jpg', '*.jpeg']),
    'textual_inversion': (SYNTH_ROOT/'textual_inversion', ['*.jpg']),
    'img2img':           (SYNTH_ROOT/'img2img',           ['*.jpg']),
    'gan_final':         (GAN_FINAL_DIR,                  ['*.png']),
    'lora':              (LORA_DIR,                       ['*.jpg']),
    'derm_s040':         (DERM_040_DIR,                   ['*.jpg']),
    'derm_s005':         (DERM_005_DIR,                   ['*.jpg']),
}
ROWS, COLS = 4, 6

# Calcular qué grillas faltan
done_grids = set()
if grids_marker.exists():
    done_grids = set(grids_marker.read_text().strip().split('\n'))

pending_grids = {k: v for k, v in GRID_SOURCES.items() if k not in done_grids}

if not pending_grids:
    print('Todas las grillas ya generadas en', EVAL_OUT)
else:
    for name, (src_dir, exts) in pending_grids.items():
        if not src_dir.exists():
            print(f'  {name}: directorio no encontrado — omitido')
            continue
        paths = [p for ext in exts for p in src_dir.glob(ext)]
        if not paths:
            continue
        sample = random.Random(42).sample(paths, min(ROWS * COLS, len(paths)))
        fig, axes = plt.subplots(ROWS, COLS, figsize=(COLS * 2, ROWS * 2))
        for ax, p in zip(axes.flat, sample):
            img = Image.open(p).convert('RGB').resize((128, 128))
            ax.imshow(img); ax.axis('off')
        for ax in axes.flat[len(sample):]:
            ax.axis('off')
        fig.suptitle(f'{name}  (muestra {len(sample)} de {len(paths)})', fontsize=11)
        fig.tight_layout()
        out_path = EVAL_OUT / f'grid_{name}.png'
        fig.savefig(out_path, dpi=120, bbox_inches='tight')
        plt.show(); plt.close(fig)
        print(f'Grilla guardada: {out_path}')
        done_grids.add(name)

    grids_marker.write_text('\n'.join(sorted(done_grids)))
    print('\nTodas las grillas generadas')

## Resumen — tabla comparativa y plots

In [ ]:
import pandas as pd

fid_data    = json.loads(fid_marker.read_text()) if fid_marker.exists() else {}
filter_data = json.loads(filter_marker.read_text()) if filter_marker.exists() else {}

EVAL_SOURCES = ['textual_inversion', 'img2img', 'gan_final', 'lora', 'derm_s040', 'derm_s005']

rows = []
for src in EVAL_SOURCES:
    row = {'Fuente': src}
    if src in fid_data and 'fid' in fid_data[src]:
        row['FID ↓']   = fid_data[src]['fid']
        row['IS ↑']    = fid_data[src]['is_mean']
        row['N synth'] = fid_data[src]['n_synth']
    if src in filter_data:
        row['Filtro %']      = filter_data[src]['pass_pct']
        row['Skin ratio']    = filter_data[src].get('skin_ratio_avg', '?')
    rows.append(row)

if rows:
    df = pd.DataFrame(rows).set_index('Fuente')
    print('\n=== Resumen de calidad ===')
    print(df.to_string())
    df.to_csv(EVAL_OUT / 'quality_summary.csv')

    numeric = df[['FID ↓', 'IS ↑']].dropna()
    if not numeric.empty:
        colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2', '#937860'][:len(numeric)]
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))

        numeric['FID ↓'].plot.bar(ax=axes[0], color=colors, rot=25)
        axes[0].set_title('FID (↓ mejor)', fontsize=11); axes[0].set_ylabel('FID')
        for bar in axes[0].patches:
            axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                         f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=8)

        numeric['IS ↑'].plot.bar(ax=axes[1], color=colors, rot=25)
        axes[1].set_title('IS (↑ mejor)', fontsize=11); axes[1].set_ylabel('IS')
        for bar in axes[1].patches:
            axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                         f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

        fig.suptitle('Calidad de imágenes sintéticas — comparación de fuentes', fontsize=12)
        fig.tight_layout()
        fig.savefig(EVAL_OUT / 'quality_summary_plot.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f'\nPlot guardado en {EVAL_OUT}/quality_summary_plot.png')
else:
    print('No hay resultados todavía — corre los pasos anteriores primero')